In [3]:
import os
import matplotlib.pyplot as plt
import numpy as np
import random
import time
import math

import scipy
from scipy import stats

In [4]:
traindata = np.loadtxt("Data_OneStepAhead/Sunspot/train7.txt")
testdata = np.loadtxt("Data_OneStepAhead/Sunspot/test7.txt") 

In [7]:
# An example of a class
class Network:
    def __init__(self, Topo, Train, Test, MaxTime, MinPer):

        self.Top = Topo  # NN topology [input, hidden, output]
        self.Max = MaxTime  # max epocs
        self.TrainData = Train
        self.TestData = Test
        self.NumSamples = Train.shape[0]

        self.lrate = 0  # will be updated later with BP call

        self.minPerf = MinPer
        # initialize weights ( W1 W2 ) and bias ( b1 b2 ) of the network
        np.random.seed()
        self.W1 = np.zeros((self.Top[0], self.Top[1])  )
        self.B1 = np.zeros(self.Top[1])    # bias first layer
        self.W2 = np.zeros((self.Top[1], self.Top[2]) )
        self.B2 = np.zeros(self.Top[2])    # bias second layer
        self.hidout = np.zeros((self.Top[1]))  # output of first hidden layer
        self.out = np.zeros((self.Top[2]))  # output last layer

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sampleEr(self, actualout):
        error = np.subtract(self.out, actualout)
        sqerror = np.sum(np.square(error)) / self.Top[2]
        return sqerror

    def ForwardPass(self, X):
        z1 = X.dot(self.W1) - self.B1
        self.hidout = self.sigmoid(z1)  # output of first hidden layer#
        z2 = self.hidout.dot(self.W2) #- self.B2
        self.out = self.sigmoid(z2)  # output second hidden layer

    def BackwardPass(self, Input, desired):

        out_delta = (desired - self.out) * (self.out * (1 - self.out))
        hid_delta = np.zeros(self.Top[2])
        hid_delta = out_delta.dot(self.W2.T) * (self.hidout * (1 - self.hidout))

    # update weights and bias
        layer = 1  # hidden to output
        for x in range(0, self.Top[layer]):
            for y in range(0, self.Top[layer + 1]):
                self.W2[x, y] += self.lrate * out_delta[y] * self.hidout[x]
        for y in range(0, self.Top[layer + 1]):
            self.B2[y] += -1 * self.lrate * out_delta[y]

        layer = 0  # Input to Hidden

        for x in range(0, self.Top[layer]):
            for y in range(0, self.Top[layer + 1]):
                self.W1[x, y] += self.lrate * hid_delta[y] * Input[x]
        for y in range(0, self.Top[layer + 1]):
            self.B1[y] += -1 * self.lrate * hid_delta[y]


    def decode(self, w):
        w_layer1size = self.Top[0] * self.Top[1]
        w_layer2size =  self.Top[1] *  self.Top[2]

        w_layer1 = w[0:w_layer1size]
        self.W1 = np.reshape(w_layer1, ( self.Top[0],  self.Top[1]))

        w_layer2 = w[w_layer1size:w_layer1size + w_layer2size]
        self.W2 = np.reshape(w_layer2, ( self.Top[1],  self.Top[2]))
        self.B1 = w[w_layer1size + w_layer2size:w_layer1size + w_layer2size +  self.Top[1]]
        self.B2 = w[w_layer1size + w_layer2size + self.Top[1]:w_layer1size + w_layer2size + self.Top[1] + self.Top[2]]
    def encode(self):
        w1 = self.W1.ravel()
        w2 = self.W2.ravel()
        w = np.concatenate([w1, w2, self.B1, self.B2])
        return w
    def net_size(self):
        return ((self.Top[0] * self.Top[1]) + (self.Top[1] * self.Top[2]) + self.Top[1] + self.Top[2])
    def evaluate_proposal(self,   w ):  # BP with SGD (Stocastic BP)

        self.decode(w)  # method to decode w into W1, W2, B1, B2.
        size = self.TrainData.shape[0]
        Input = np.zeros((1, self.Top[0]))  # temp hold input
        fx = np.zeros(size)



        for pat in range(0, size):
            Input[:] =  self.TrainData[pat, 0:self.Top[0]]
            self.ForwardPass(Input)
            fx[pat] = self.out
        return fx

    def test_proposal(self,  w):  # BP with SGD (Stocastic BP)

        self.decode(w)  # method to decode w into W1, W2, B1, B2.

        size = self.TestData.shape[0]
        Input = np.zeros((1, self.Top[0]))  # temp hold input
        Desired = np.zeros((1, self.Top[2]))
        fx = np.zeros(size)

        sse = 0

        for pat in range(0, size):
            Input[:] = self.TestData[pat, 0:self.Top[0]]
            Desired[:] = self.TestData[pat, self.Top[0]:]
            self.ForwardPass(Input)
            fx[pat] = self.out
            sse = sse + self.sampleEr(Desired)

        rmse = np.sqrt(sse / size)


        return [fx,rmse]


In [8]:
input = 7  # max input
hidden = 5
output = 1
baseNet = [input, hidden, output]
secondnet = [input, hidden+1, output]
mtaskNet = np.array([baseNet, secondnet])
dims = len(mtaskNet)

In [9]:
Net = Network(mtaskNet[0], traindata, testdata, 0.1, 0.0002)
netsize = Net.net_size()
w = np.random.randn(netsize)
w
Net.evaluate_proposal(w)

array([0.40110544, 0.40338843, 0.40408856, 0.39933651, 0.39930223,
       0.38745784, 0.3739554 , 0.3607752 , 0.33352754, 0.30895901,
       0.27452391, 0.24061213, 0.21554531, 0.1942255 , 0.18084341,
       0.16910634, 0.16103507, 0.1602384 , 0.1569656 , 0.1525162 ,
       0.15254208, 0.15316454, 0.15345751, 0.1527951 , 0.15283459,
       0.15723419, 0.15755362, 0.1591922 , 0.16326896, 0.16679441,
       0.17337793, 0.17586209, 0.18126885, 0.19131421, 0.18896094,
       0.18540664, 0.18708855, 0.19229433, 0.19560458, 0.19700062,
       0.21101604, 0.22398645, 0.23209717, 0.24943577, 0.26132873,
       0.26722321, 0.2767604 , 0.2905654 , 0.30361044, 0.3082121 ,
       0.31831621, 0.33081645, 0.34194844, 0.34370765, 0.3455102 ,
       0.35401021, 0.3637015 , 0.36948317, 0.37448122, 0.38456751,
       0.39216911, 0.38764979, 0.38501728, 0.38564513, 0.37760371,
       0.37030753, 0.36250542, 0.35025914, 0.33066096, 0.31472337,
       0.30684936, 0.29470565, 0.28184214, 0.2730757 , 0.26131

In [10]:
Net1 = Network(mtaskNet[1], traindata, testdata, 0.1, 0.0002)
netsize1 = Net1.net_size()
v = np.random.normal(0, 1, netsize1-netsize)
w_proposal = np.concatenate((w, v), axis = None)
#w_pro = np.random.randn(netsize1)
w_proposal
Net1.evaluate_proposal(w_proposal)

array([0.47707988, 0.47551345, 0.47552806, 0.46793073, 0.46686269,
       0.47083779, 0.45494217, 0.45271301, 0.44155345, 0.43662016,
       0.44573017, 0.43664726, 0.43802967, 0.44648623, 0.4703353 ,
       0.47179986, 0.43072969, 0.45096387, 0.49460889, 0.47854815,
       0.47704625, 0.50478568, 0.52315449, 0.51466974, 0.48466897,
       0.50701256, 0.51673623, 0.49953677, 0.50267895, 0.50066314,
       0.51833021, 0.5020095 , 0.45184403, 0.46957455, 0.50086302,
       0.49003856, 0.47699018, 0.48890486, 0.51576388, 0.50014793,
       0.49091923, 0.50593497, 0.48536555, 0.47762268, 0.49359611,
       0.49490618, 0.48312103, 0.47747207, 0.49086241, 0.4942328 ,
       0.4812527 , 0.47147647, 0.4779837 , 0.48890608, 0.48228545,
       0.47917952, 0.48435935, 0.48985248, 0.47963594, 0.46930302,
       0.47539403, 0.47781085, 0.47096846, 0.47452567, 0.47197842,
       0.46001453, 0.45717207, 0.46693884, 0.4729615 , 0.46360066,
       0.46380334, 0.47180198, 0.47045889, 0.46589284, 0.45497

In [17]:
class MCMC:
	def __init__(self, samples, traindata, testdata, minPerf, mtaskNet):
		self.samples = samples  
		self.mtaskNet = mtaskNet  # mtaskNet = np.array([baseNet, secondnet]) -> to access each nn, mtaskNet[dims]
		self.traindata = traindata  #
		self.testdata = testdata 
		self.minPerf = minPerf
		# ----------------

	def rmse(self, predictions, targets):
		return np.sqrt(((predictions - targets) ** 2).mean())

	def likelihood_func(self, neuralnet, data, w, tausq, dims):
		topology = self.mtaskNet[dims]
		y = data[:, topology[0]]
		fx = neuralnet.evaluate_proposal(w)
		rmse = self.rmse(fx, y)
		n = y.shape[0]

		loss =( -(n/2) * np.log(2 * math.pi * tausq)) -( (1/(2*tausq)) * np.sum(np.square(y - fx))) #tausq is variance of error terms
		return [loss, fx, rmse]

	def prior_likelihood(self, sigma_squared, nu_1, nu_2, w, tausq, model_prior, dims):
		topology = self.mtaskNet[dims]
		h = topology[1]  # number hidden neurons
		d = topology[0]  # number input neurons
		part1 = -1 * ((d * h + h + 2) / 2) * np.log(sigma_squared)
		part2 = -1 / (2 * sigma_squared) * (sum(np.square(w)))
		part3 = -1 * np.log(model_prior)
		log_loss = part1 + part2 + part3 - (1 + nu_1) * np.log(tausq) - (nu_2 / tausq)
		return log_loss

	def v_jump(self, tausq, v, size_diff): #choose tausq well for better proposal
		log_v =( -(size_diff/2) * np.log(2 * math.pi * tausq)) -( (1/(2*tausq)) * np.sum(np.square(v))) #tausq is variance of error terms
		return log_v

	def sampler(self, dims1, dims2):
		#----- Model 1
		netw_1 = self.mtaskNet[dims1]		
		Net1 = Network(netw_1, self.traindata, self.testdata, 0.1, 0.0002)
		#--- Initial w
		w_size1 = Net1.net_size()
		w_net1 = np.random.normal(0,1,w_size1)		
		y_test_1 = self.testdata[:, netw_1[0]]
		y_train_1 = self.traindata[:, netw_1[0]]

		#----- Model 2
		netw_2 = self.mtaskNet[dims2]
		Net2 = Network(netw_2, self.traindata, self.testdata, 0.1, 0.0002)
		#--- Initial w
		w_size2 = Net2.net_size()
		w_net2 = np.random.normal(0,1,w_size2)
		y_test_2 = self.testdata[:, netw_2[0]]
		y_train_2 = self.traindata[:, netw_2[0]]
		#----- Initialize MCMC
		samples = self.samples
		testsize = self.testdata.shape[0]
		trainsize = self.traindata.shape[0]
		fxtrain_samples = []
		fxtest_samples = []
		rmse_train = []
		rmse_test = []
		pos_w = []
		#--- HYPERPARAMETERS
		sigma_squared = 25
		nu_1 = 0
		nu_2 = 0
		model_prior = 1/2 #for both models 1 and 2
		is_in_lower_dimension = True
		
		naccept = 0
		for i in range(samples -1):
			if is_in_lower_dimension: #jump up dimension: netw1 < netw2
				#--- Propose error term
				pred_train = Net1.evaluate_proposal(w_net1)
				eta = np.log(np.var(pred_train - y_train_1))
				tau_pro = np.exp(eta)
				#--- Calc current prior and likelihood
				[likelihood_current, pred_train, rmse_train_current] = self.likelihood_func(Net1, self.traindata, w_net1, tau_pro, dims1) #w here used for bnn
				[pred_test_current, rmse_test_current] = Net1.test_proposal(w_net1)	
				prior_current = self.prior_likelihood(sigma_squared, nu_1, nu_2, w_net1, tau_pro, model_prior, dims1)
				#--- Propose jump vector v
				v = np.random.normal(0, 1, w_size2-w_size1)
				#--- Propose w
				w_proposal = np.concatenate((w_net1, v), axis = None)
				#--- Calc proposed prior and likelihood
				[likelihood_proposal, pred_train_proposal, rmse_train_proposal] = self.likelihood_func(Net2, self.traindata, w_proposal, tau_pro, dims2)
				[pred_test_proposal, rmse_test_proposal] = Net2.test_proposal(w_proposal)				
				prior_proposal = self.prior_likelihood(sigma_squared, nu_1, nu_2, w_proposal, tau_pro, model_prior, dims2)
				log_v = self.v_jump(tau_pro, v, w_size2-w_size1)
				#--- Calc log posterior 
				log_posterior = (prior_proposal + likelihood_proposal) - (prior_current + likelihood_current + log_v) 
				try:
					mh_prob = min(1, math.exp(log_posterior))

				except OverflowError as e:
					mh_prob = 1

				u = random.uniform(0, 1)

				if u < mh_prob:
					# ACCEPT
					naccept += 1
					likelihood_current = likelihood_proposal
					prior_current = prior_proposal            
					w_net2 = w_proposal #assign proposed w to w which now has higher dim
					pos_w.append(w_net2)
					is_in_lower_dimension = False


					fxtrain_samples.append(pred_train_proposal)
					fxtest_samples.append(pred_test_proposal)
					rmse_train.append(rmse_train_proposal)
					rmse_test.append(rmse_test_proposal)
				else:
					fxtrain_samples.append(pred_train)
					fxtest_samples.append(pred_test_current)
					rmse_train.append(rmse_train_current)
					rmse_test.append(rmse_test_current)
				print(i, 'jump from model 1 with size', w_size1,'to model 2 with size', w_size2, math.exp(log_posterior))

			else: #	
				#--- Propose error term
				pred_train = Net2.evaluate_proposal(w_net2)
				eta = np.log(np.var(pred_train - y_train_1))
				tau_pro = np.exp(eta)
				#--- Calc current prior and likelihood

				[likelihood_current, pred_train, rmse_train_current] = self.likelihood_func(Net2, traindata, w_net2, tau_pro, dims2) #w here used for bnn
				[pred_test_current, rmse_test_current] = Net2.test_proposal(w_net2)	
				prior_current = self.prior_likelihood(sigma_squared, nu_1, nu_2, w_net2, tau_pro, model_prior, dims2)
				#--- Propose jump vector v
				v = np.random.normal(0, 1, w_size2-w_size1)
				#--- Propose w
				w_proposal = w_net2[0:w_size1] #propose w_down that has lower dim
				#--- Calc proposed prior and likelihood
				[likelihood_proposal, pred_train_proposal, rmse_train_proposal] = self.likelihood_func(Net1, self.traindata, w_proposal, tau_pro, dims1)
				[pred_test_proposal, rmse_test_proposal] = Net1.test_proposal(w_proposal)	
				prior_proposal = self.prior_likelihood(sigma_squared, nu_1, nu_2, w_proposal, tau_pro, model_prior, dims1)
				log_v = self.v_jump(tau_pro, v, w_size2-w_size1)
				#--- Calc log posterior 

				log_posterior_down = (prior_proposal + likelihood_proposal + log_v) - (prior_current + likelihood_current) 
				try:
					mh_prob_down = min(1, (log_posterior_down))
				except OverflowError as e:
					mh_prob_down = 1
				u_down = random.uniform(0, 1)
				if u_down < mh_prob_down:
					naccept += 1
					likelihood_current = likelihood_proposal
					prior_current = prior_proposal
					w_net1 = w_proposal
					pos_w.append(w_net1)					
					is_in_lower_dimension = True

					fxtrain_samples.append(pred_train_proposal)
					fxtest_samples.append(pred_test_proposal)
					rmse_train.append(rmse_train_proposal)
					rmse_test.append(rmse_test_proposal)
				else:
					fxtrain_samples.append(pred_train)
					fxtest_samples.append(pred_test_current)
					rmse_train.append(rmse_test_current)
					rmse_test.append(rmse_test_current)
				print(i, 'jump from model 2 with size', w_size2,'to model 1 with size', w_size1)
			accept_ratio = naccept / (samples)
			print(np.shape(pos_w), '{:.1%}'.format(accept_ratio), 'is accepted')
		return (pos_w, fxtrain_samples, fxtest_samples, rmse_train, rmse_test, accept_ratio)


In [21]:
mcmc = MCMC(5, traindata, testdata, 0.002, mtaskNet)
mcmc.sampler(0,1)

0 jump from model 1 with size 46 to model 2 with size 55 3.2306365737843705e+162
(1, 55) 20.0% is accepted
1 jump from model 2 with size 55 to model 1 with size 46
(1, 55) 20.0% is accepted
2 jump from model 2 with size 55 to model 1 with size 46
(1, 55) 20.0% is accepted
3 jump from model 2 with size 55 to model 1 with size 46
(1, 55) 20.0% is accepted


([array([-0.02853356, -0.42305863, -0.96794173,  0.8629012 ,  0.1106117 ,
          1.65304108, -0.44838306, -0.15094208,  0.48815305, -0.28514064,
         -1.96440697, -0.32597795,  0.57561197,  0.02962736,  0.1313962 ,
         -0.54346879,  0.30059026,  0.34081383,  0.08148377,  0.00349568,
          0.41125151, -0.55197689,  0.57576987, -1.79887882, -0.69629966,
         -0.37321096,  0.31992933,  0.69811318,  0.04888983,  0.32585909,
          0.33456248,  1.85138848,  0.86760993,  0.63174218, -0.90773903,
         -1.30964769,  0.72948265,  1.20592811,  0.82214391,  0.17550489,
         -0.21891676,  1.81582713, -0.80311175, -0.51561221,  1.20356327,
         -0.23526151, -1.11927718,  0.78584766, -1.06263701,  0.40290024,
         -1.0458848 , -0.59629411, -0.11604827,  0.11887423,  0.30902357])],
 [array([0.43926463, 0.43914503, 0.43853182, 0.44085727, 0.44060997,
         0.44353109, 0.44983965, 0.45116241, 0.46136068, 0.46668318,
         0.47547511, 0.48909305, 0.49619867, 

In [9]:
[pos_w, fxtrain_samples, fxtest_samples, rmse_train, rmse_test, accept_ratio] = mcmc.sampler(0,1)
fx_mu = np.mean(fxtest_samples, axis=0)
#np.shape(fxtrain_samples)
fx_mu

0 jump from model 1 with size 46 to model 2 with size 37 (1, 37) 20.0% is accepted
1 jump from model 1 with size 46 to model 2 with size 37 (1, 37) 20.0% is accepted
2 jump from model 1 with size 46 to model 2 with size 55 (1, 37) 20.0% is accepted
3 jump from model 1 with size 46 to model 2 with size 37 (2, 37) 40.0% is accepted


array([0.83356637, 0.83651351, 0.83729304, 0.83431551, 0.82965127,
       0.82271682, 0.81624103, 0.81367337, 0.81258875, 0.81271461,
       0.80955582, 0.80495116, 0.79865887, 0.78934761, 0.77917556,
       0.76651112, 0.75660706, 0.74619904, 0.73609623, 0.72698496,
       0.71968083, 0.71724509, 0.71692055, 0.71704614, 0.71695859,
       0.71417915, 0.71036153, 0.70635307, 0.7026518 , 0.70094515,
       0.69894556, 0.6984926 , 0.69945539, 0.7000121 , 0.70089175,
       0.69993676, 0.69907519, 0.69814096, 0.69770252, 0.69853952,
       0.7004426 , 0.70462943, 0.71130608, 0.72261701, 0.73633699,
       0.75202672, 0.76685536, 0.78183433, 0.79599488, 0.80849872,
       0.81848476, 0.82582167, 0.83197299, 0.83582237, 0.83906459,
       0.84121761, 0.84387634, 0.8462544 , 0.84862293, 0.84989358,
       0.8508128 , 0.85126303, 0.85033932, 0.84907047, 0.84573749,
       0.84313061, 0.83996991, 0.8360778 , 0.83223328, 0.82785889,
       0.82489077, 0.82241622, 0.81791912, 0.81237787, 0.80580